# How to run mass parallelized simulations with Slurm
To gain the ability to run a massive amount of simulations parllelized over multiple machines and cores we intruced Slurm. </br>
The following is meant to show how to prepare a simulation for Slurm.

### First we must import the necessary packages

In [1]:
from hera.utils.logging import initialize_logging, with_logger
from hera.utils.slurm import prepareSlurmScriptExecution
import hera.toolkit
import json
import numpy as np
initialize_logging(
    with_logger("hera.simulations", handlers=['console'], level='DEBUG', propagate=False),
    with_logger("hera.bin", handlers=['console'], level='DEBUG', propagate=False)
)

### Loadng the OpenFOAM simulation toolkit 

In [3]:
PROJECT_NAME = "pipe_slurm"
PROCESSOR_PER_SIM = 80

tk = hera.toolkit.ToolkitHome().getToolkit(hera.toolkit.ToolkitHome.SIMULATIONS_OPENFOAM, PROJECT_NAME)

Success
INFO    : project.py/__init__(236) Initializing with logger pipe_slurm
INFO    : project.py/__init__(245) Attempting to get default directory from the disk
DEBUG   : jsonutils.py/ConfigurationToJSON(65) Processing {}
DEBUG   : jsonutils.py/ConfigurationToJSON(65) Processing {}
DEBUG   : jsonutils.py/ConfigurationToJSON(65) Processing {}
DEBUG   : jsonutils.py/ConfigurationToJSON(65) Processing {}


/home/liora/Development/pyhera/hera/utils/freeCAD.py:10: UserWarning: freecad is not installed. some features will not work.
  warnings.warn("freecad is not installed. some features will not work.")


### Create variations file for the simulation

In [3]:
variation=[]

variation.append({"workflow.nodes.Parameters.Execution.input_parameters.pipe_radius": np.linspace(0.2, 0.3, 3).tolist()})
variation.append({"workflow.nodes.Parameters.Execution.input_parameters.velocity" : np.linspace(1, 2, 3).tolist()})    
variation.append({"workflow.nodes.Parameters.Execution.input_parameters.y_plus" : np.linspace(20, 70, 3).tolist()})    

# 3 * 3 * 3 = 27 variations

with open("variation.json", "w") as f:
    json.dump(variation, f)


### Generate the slurm scipt
Notice that it takes some time to generate all the variations

In [ ]:
tk.prepareSlurmWorkflowExecution(
    baseConfiguration="pipe_0000",
    jsonVariations="variation.json",
    slurmExecutionFileName="submit_all.sh",
    caseListFileName="cases.txt",
    allocateProcessorsPerRun=PROCESSOR_PER_SIM,
    memoryInGB=None,
    exclusive=False,
    addAllRun=False
)


### Now we must run the following command in terminal
```bash
sbatch submit_all.sh
```
You can look at the progress using:
```bash
squeue
```
if it not closing it means it is unresponsive!

### We can also run a script per folder
Notice that this doesn't run inside a project, this means you have to provide the exact path for the execution script

In [ ]:
script = f"""
cd "$dir" || {{ echo "Directory $dir not found"; exit 1; }}
mpirun -n {PROCESSOR_PER_SIM} simpleFoam -parallel -postProcess -latestTime -func yPlus
"""
prepareSlurmScriptExecution(scriptPath=None, script=script,
                              slurmExecutionFilePath="submit_yplus_all.sh",
                              jobDirListFilePath="cases.txt", # exists from previous cell
                              allocateProcessorsPerRun=PROCESSOR_PER_SIM,
                              memoryInGB=None,
                              jobName="yplus_calc",
                              exclusive=False)


writing to /raid/users/liora/Development/pyhera/hera/doc/jupyter/User/utils/submit_yplus_all.sh
